# Fraud Detection Model Training

**Purpose.** Train the `RandomForestClassifier` that backs the Enterprise Real-Time
Fraud Detection Engine and persist it as `models/fraud_model.joblib`, the artifact
loaded by `app/model_utils.py` at API start-up.

**Why synthetic data?** Real card-transaction datasets are regulated (PCI-DSS, GDPR)
and cannot be committed to a public repository. `sklearn.datasets.make_classification`
gives us a fully reproducible, licence-free dataset with a controllable class
imbalance, which is enough to exercise the complete train -> serialise -> serve
-> test pipeline. The feature engineering below rescales the synthetic columns
into realistic ranges so the API examples read like real transactions.

The feature column order defined here (`amount`, `distance_from_home`, `use_chip`)
is a contract: `FraudDetectionModel.predict` builds its feature vector in exactly
this order.

In [1]:
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
FEATURE_COLUMNS = ["amount", "distance_from_home", "use_chip"]

In [2]:
X, y = make_classification(
    n_samples=5000,
    n_features=3,
    n_informative=3,
    n_redundant=0,
    weights=[0.95, 0.05],
    random_state=RANDOM_STATE,
)

df = pd.DataFrame(X, columns=FEATURE_COLUMNS)
df["is_fraud"] = y


def min_max_scale(series: pd.Series, low: float, high: float) -> pd.Series:
    """Linearly rescale a series into the closed interval [low, high]."""
    s_min, s_max = series.min(), series.max()
    return low + (series - s_min) * (high - low) / (s_max - s_min)


# Rescale synthetic features into realistic transaction ranges.
df["amount"] = min_max_scale(df["amount"], 0.0, 5000.0).round(2)
df["distance_from_home"] = min_max_scale(df["distance_from_home"], 0.0, 200.0).round(2)
# Binarise chip usage: values above the median become 1 (chip used), else 0.
df["use_chip"] = (df["use_chip"] > df["use_chip"].median()).astype(int)

print(df.describe().T)
print()
print("Class balance:")
print(df["is_fraud"].value_counts(normalize=True).rename("fraction"))
df.head()

                     count         mean         std  min        25%      50%  \
amount              5000.0  2617.555822  741.051431  0.0  2110.7175  2655.96   
distance_from_home  5000.0   102.983726   25.429941  0.0    86.6700   102.78   
use_chip            5000.0     0.500000    0.500050  0.0     0.0000     0.50   
is_fraud            5000.0     0.054400    0.226828  0.0     0.0000     0.00   

                         75%     max  
amount              3151.935  5000.0  
distance_from_home   120.125   200.0  
use_chip               1.000     1.0  
is_fraud               0.000     1.0  

Class balance:
is_fraud
0    0.9456
1    0.0544
Name: fraction, dtype: float64


,amount,distance_from_home,use_chip,is_fraud
0,3057.10,142.63,1,1
1,3042.99,134.48,1,0
2,1669.44,71.48,0,0
3,2283.18,99.93,0,0
4,1029.55,73.07,0,0


## Why a 95 / 5 class imbalance?

Card fraud is a rare event: published industry figures put the fraudulent share
of transactions well under 1 %. Training on a perfectly balanced dataset would
make the model over-confident about fraud and produce a flood of false positives
in production. A 5 % positive rate is a pragmatic compromise for a synthetic
dataset: it is imbalanced enough that accuracy alone is misleading (a constant
"legit" predictor scores ~95 %), so we must look at per-class precision, recall
and F1 in the classification report, yet it still leaves enough positive samples
(~250) for a random forest to learn a stable decision boundary without
resampling tricks.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    df[FEATURE_COLUMNS],
    df["is_fraud"],
    test_size=0.2,
    stratify=df["is_fraud"],
    random_state=RANDOM_STATE,
)

model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
model.fit(X_train.to_numpy(), y_train.to_numpy())

print(f"Trained on {len(X_train)} rows, holding out {len(X_test)} for evaluation.")

Trained on 4000 rows, holding out 1000 for evaluation.


In [4]:
y_pred = model.predict(X_test.to_numpy())
print(classification_report(y_test, y_pred, target_names=["legit", "fraud"], digits=4))

              precision    recall  f1-score   support

       legit     0.9520    0.9852    0.9683       946
       fraud     0.3333    0.1296    0.1867        54

    accuracy                         0.9390      1000
   macro avg     0.6427    0.5574    0.5775      1000
weighted avg     0.9186    0.9390    0.9261      1000



In [5]:
os.makedirs("../models", exist_ok=True)
artifact_path = os.path.join("..", "models", "fraud_model.joblib")
joblib.dump(model, artifact_path)

size_kb = os.path.getsize(artifact_path) / 1024
print(f"Saved model to {artifact_path} ({size_kb:.1f} KB)")

# Round-trip sanity check: reload and score one obviously risky transaction.
reloaded = joblib.load(artifact_path)
sample = np.array([[4800.0, 180.0, 0]])
print("Reloaded prediction for [4800.0, 180.0, 0]:", int(reloaded.predict(sample)[0]))

Saved model to ..\models\fraud_model.joblib (3570.5 KB)
Reloaded prediction for [4800.0, 180.0, 0]: 0
